# LEDGAR Clause Classification Pipeline

Coursework support notebook for contract clause classification using LexGLUE LEDGAR.

Academic boundary: this notebook prepares code, metrics, plots, predictions, and saved outputs only. It does not write report text, invent results, or provide final conclusions.


## 1. Setup and Imports

Create the expected project folders and import the libraries used by the pipeline.


In [ ]:
from pathlib import Path
import importlib.util
import json
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd()).resolve()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

try:
    from datasets import load_dataset
    print("datasets package available")
except Exception as exc:
    print(f"WARNING: datasets import failed: {type(exc).__name__}: {exc}")

TRANSFORMERS_AVAILABLE = importlib.util.find_spec("transformers") is not None
if TRANSFORMERS_AVAILABLE:
    import transformers
    print(f"transformers package available: {transformers.__version__}")
else:
    print("transformers package is not installed. Transformer cells are guarded and disabled by default.")

from preprocess_ledgar import (
    create_eda_outputs,
    create_label_mapping,
    dataset_overview,
    ensure_project_dirs,
    export_raw_hf_splits,
    filter_splits_to_labels,
    load_ledgar_dataset,
    load_processed_splits,
    print_filtered_summary,
    print_label_inspection,
    save_label_artifacts,
    save_processed_splits,
    save_selection_artifacts,
    select_labels,
    standardise_splits,
    warn_messages,
    apply_label_mapping,
)
from train_classical import (
    create_error_analysis,
    create_final_comparison,
    load_label_mapping,
    run_baselines,
    run_classical_models,
)
from train_transformer import run_transformer_experiments, transformers_available
from ledgar_pipeline.config import get_config
from ledgar_pipeline.wandb_tracking import log_preprocessing_run, require_wandb, wandb_is_installed

CONFIG = get_config(PROJECT_ROOT)
CONFIG.paths.ensure_dirs()
paths = ensure_project_dirs(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")
for name, path in paths.items():
    if name != "root":
        print(f"{name}: {path}")


## 2. GPU / CUDA Check

CUDA is only relevant for transformer fine-tuning. TF-IDF, Logistic Regression, and Linear SVM are CPU-based.


In [ ]:
print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
print(f"torch.version.cuda: {torch.version.cuda}")
print(f"torch.cuda.device_count(): {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print(f"torch.cuda.get_device_name(0): {torch.cuda.get_device_name(0)}")
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
    print("WARNING: CUDA is unavailable. Transformer fine-tuning will run on CPU if enabled and may be slow.")

print(f"Selected device: {DEVICE}")
print("TF-IDF, Logistic Regression, and Linear SVM are CPU-based.")
print("CUDA mainly matters for BERT/LegalBERT transformer fine-tuning.")


## 3. W&B Tracking Check

W&B logging is compulsory for this pipeline. Training/evaluation cells will fail clearly if W&B is unavailable, offline, disabled, or not authenticated.


In [ ]:
print(f"wandb installed: {wandb_is_installed()}")
print(f"W&B project: {CONFIG.wandb.project}")
print(f"W&B entity: {CONFIG.wandb.entity}")
print(f"W&B mode: {CONFIG.wandb.mode}")
require_wandb(CONFIG.wandb)
print("W&B is required and online mode is enforced. If a later W&B run fails, run `wandb login` or set WANDB_API_KEY.")


## 4. Reference Notebook Style Check

Inspect local reference notebooks only for style signals. Do not copy their content, results, or exercises.


In [ ]:
reference_dir = paths["reference_notebooks"]
style_rows = []

if reference_dir.exists():
    for notebook_path in sorted(reference_dir.glob("*.ipynb")):
        with notebook_path.open("r", encoding="utf-8") as f:
            nb = json.load(f)
        notebook_cells = nb.get("cells", [])
        markdown_cells = [cell for cell in notebook_cells if cell.get("cell_type") == "markdown"]
        code_cells = [cell for cell in notebook_cells if cell.get("cell_type") == "code"]
        heading_count = 0
        todo_count = 0
        code_lengths = []
        for cell in markdown_cells:
            source = "".join(cell.get("source", []))
            todo_count += source.lower().count("todo")
            heading_count += sum(1 for line in source.splitlines() if line.strip().startswith("#"))
        for cell in code_cells:
            source = "".join(cell.get("source", []))
            code_lengths.append(len([line for line in source.splitlines() if line.strip()]))
        style_rows.append({
            "notebook": notebook_path.name,
            "markdown_cells": len(markdown_cells),
            "code_cells": len(code_cells),
            "heading_count": heading_count,
            "todo_mentions": todo_count,
            "avg_code_lines": round(float(np.mean(code_lengths)), 1) if code_lengths else 0,
        })
else:
    print("No reference_notebooks directory found. Continuing without local style inspection.")

if style_rows:
    display(pd.DataFrame(style_rows))

print("Style summary for this notebook:")
print("- numbered headings")
print("- short markdown explanations")
print("- compact code cells")
print("- TODO prompts for student inspection")
print("- clear displayed tables and saved outputs")


## 5. Load LexGLUE LEDGAR

Load the main dataset from Hugging Face. If loading fails, print the error clearly and attempt the configured JSONL fallback.


In [ ]:
ds, dataset_metadata = load_ledgar_dataset(PROJECT_ROOT)
overview = dataset_overview(ds)

print(f"Dataset source: {dataset_metadata['dataset_source']}")
print(f"Available splits: {overview['splits']}")
print(f"Rows per split: {overview['rows_per_split']}")
print("Dataset features:")
print(overview["features"])

features = dataset_metadata.get("features")
label_feature = features.get("label") if hasattr(features, "get") else None
print("Label feature information:")
print(label_feature)
if hasattr(label_feature, "names"):
    print(f"Number of label names: {len(label_feature.names)}")
    print(f"First 10 label names: {label_feature.names[:10]}")

print("First 3 training examples:")
display(pd.DataFrame(overview["train_examples"]))


## 6. Export Raw Hugging Face Splits to JSONL

Save the official Hugging Face splits as raw JSONL files.


In [ ]:
raw_paths = {}
if dataset_metadata["dataset_source"] == "huggingface":
    raw_paths = export_raw_hf_splits(ds, paths["raw_lexglue"])
    for split, path in raw_paths.items():
        print(f"Saved raw {split}: {path}")
else:
    print("Fallback dataset is in use, so raw Hugging Face split export was skipped.")


## 7. Standardise Schema

Convert dataset rows to: `text`, `label`, `label_id`, `source_dataset`, `source_id`, `split`.

Preprocessing keeps legal wording intact: whitespace is normalised only.


In [ ]:
standardised_splits = standardise_splits(ds, dataset_metadata)

for split, df in standardised_splits.items():
    print(f"{split}: {df.shape}")
    print(df.columns.tolist())

display(standardised_splits["train"].head(3))


## 8. Label Inspection

Inspect label frequencies before selecting a modelling subset.

TODO: Review the top and bottom labels before changing the label-selection settings.


In [ ]:
print_label_inspection(standardised_splits)
label_artifacts = save_label_artifacts(standardised_splits, paths["outputs"])
print(label_artifacts)


## 9. Label Filtering

Default setting: use the top 20 labels by training frequency. Manual labels are matched case-insensitively if that mode is selected.


In [ ]:
LABEL_SELECTION_MODE = "top_n"  # options: "all", "top_n", "manual"
TOP_N_LABELS = 20
MIN_EXAMPLES_PER_LABEL = 30
MANUAL_SELECTED_LABELS = []

selected_labels, selection_warnings = select_labels(
    standardised_splits["train"],
    mode=LABEL_SELECTION_MODE,
    top_n=TOP_N_LABELS,
    manual_labels=MANUAL_SELECTED_LABELS,
    min_examples_per_label=MIN_EXAMPLES_PER_LABEL,
)
warn_messages(selection_warnings)

filtered_splits = filter_splits_to_labels(standardised_splits, selected_labels)
label_to_id, id_to_label = create_label_mapping(selected_labels)
processed_splits = apply_label_mapping(filtered_splits, label_to_id)
selection_artifacts = save_selection_artifacts(selected_labels, label_to_id, id_to_label, paths["outputs"])

print_filtered_summary(processed_splits, selected_labels)
print(selection_artifacts)


## 10. Save Processed Splits

Save processed train, validation, and test splits as JSONL using the standard schema.


In [ ]:
processed_paths = save_processed_splits(processed_splits, paths["processed"])
for split, path in processed_paths.items():
    print(f"Saved processed {split}: {path}")

schema_check = {split: df.columns.tolist() for split, df in processed_splits.items()}
print(schema_check)


## 11. Exploratory Data Analysis

Create simple tables and figures for class distribution and text length.

TODO: Inspect these outputs before deciding whether to adjust label filtering.


In [ ]:
eda_outputs = create_eda_outputs(processed_splits, paths["outputs"], paths["figures"])
print(eda_outputs)

combined_processed = pd.concat(processed_splits.values(), ignore_index=True)
class_distribution = (
    combined_processed.groupby(["split", "label"])
    .size()
    .reset_index(name="count")
    .sort_values(["split", "count"], ascending=[True, False])
)
display(class_distribution.head(30))

length_stats = combined_processed.assign(
    word_count=combined_processed["text"].str.split().str.len(),
    character_count=combined_processed["text"].str.len(),
)[["word_count", "character_count"]].describe()
display(length_stats)

examples = pd.read_json(paths["outputs"] / "example_clauses.jsonl", lines=True)
display(examples.head(10))

metadata_paths = {
    **selection_artifacts,
    **eda_outputs,
    "label_names": paths["outputs"] / "label_names.txt",
    "label_counts": paths["outputs"] / "label_counts.json",
}
log_preprocessing_run(
    raw_paths=raw_paths,
    processed_paths=processed_paths,
    metadata_paths=metadata_paths,
    sample_frames=processed_splits,
    config={"pipeline": CONFIG.to_dict(), "dataset_source": dataset_metadata["dataset_source"]},
    wandb_config=CONFIG.wandb,
)
print("Preprocessing data, metadata, samples, and figures logged to W&B.")


## 12. Evaluation Helpers

Primary metric: macro-F1. Secondary metrics include accuracy, weighted-F1, macro precision, macro recall, per-class F1, and confusion matrix.


In [ ]:
train_df, validation_df, test_df = load_processed_splits(paths["processed"]).values()
label_to_id, id_to_label = load_label_mapping(paths["outputs"])

print(f"Train rows: {len(train_df)}")
print(f"Validation rows: {len(validation_df)}")
print(f"Test rows: {len(test_df)}")
print(f"Labels: {len(id_to_label)}")
print("Evaluation helpers are imported from src/evaluate.py and src/train_classical.py.")


## 13. Baseline Models

Run random and majority-class baselines on validation and test splits. Test predictions are saved as JSONL.


In [ ]:
baseline_results = run_baselines(
    train_df,
    validation_df,
    test_df,
    outputs_dir=paths["outputs"],
    predictions_dir=paths["predictions"],
    id_to_label=id_to_label,
    reset_results=True,
    wandb_config=CONFIG.wandb,
)

display(pd.DataFrame([
    {
        "model_name": row["model_name"],
        "split": row["split"],
        **row["metrics"],
    }
    for row in baseline_results
]))


## 14. Classical TF-IDF Models

Train TF-IDF + Logistic Regression and TF-IDF + Linear SVM. Select the best configuration using validation macro-F1, then evaluate on test.


In [ ]:
RUN_CLASSICAL_MODELS = True

if RUN_CLASSICAL_MODELS:
    classical_results = run_classical_models(
        train_df,
        validation_df,
        test_df,
        outputs_dir=paths["outputs"],
        predictions_dir=paths["predictions"],
        figures_dir=paths["figures"],
        models_dir=paths["models_trained_classical"],
        checkpoints_dir=paths["checkpoints_classical"],
        id_to_label=id_to_label,
        wandb_config=CONFIG.wandb,
    )
    summary_rows = []
    for model_name, result_bundle in classical_results.items():
        test_result = result_bundle["test_result"]
        summary_rows.append({"model_name": model_name, **test_result["metrics"]})
    display(pd.DataFrame(summary_rows).sort_values("macro_f1", ascending=False))
else:
    print("Classical model training skipped because RUN_CLASSICAL_MODELS is False.")


## 15. Transformer Experiment Template

Transformer fine-tuning is disabled by default. Change `RUN_TRANSFORMERS` only when you are ready for a long training run.


In [ ]:
RUN_TRANSFORMERS = False
TRANSFORMER_MAX_LENGTH = 256
TRANSFORMER_BATCH_SIZE = 8
TRANSFORMER_EPOCHS = 3

print(f"RUN_TRANSFORMERS: {RUN_TRANSFORMERS}")
print(f"transformers installed: {transformers_available()}")
print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
print(f"fp16 will be set to torch.cuda.is_available(): {torch.cuda.is_available()}")

if RUN_TRANSFORMERS and torch.cuda.is_available():
    torch.cuda.empty_cache()
elif RUN_TRANSFORMERS and not torch.cuda.is_available():
    print("WARNING: RUN_TRANSFORMERS is True but CUDA is unavailable. Training may be very slow.")

transformer_results = run_transformer_experiments(
    train_df,
    validation_df,
    test_df,
    outputs_dir=paths["outputs"],
    predictions_dir=paths["predictions"],
    models_dir=paths["models_trained_transformers"],
    checkpoints_dir=paths["checkpoints_transformers"],
    id_to_label=id_to_label,
    wandb_config=CONFIG.wandb,
    run_transformers=RUN_TRANSFORMERS,
    max_length=TRANSFORMER_MAX_LENGTH,
    batch_size=TRANSFORMER_BATCH_SIZE,
    epochs=TRANSFORMER_EPOCHS,
)

if transformer_results:
    display(pd.DataFrame([{ "model_name": row["model_name"], **row["metrics"] } for row in transformer_results]))


## 16. Final Results Comparison

Combine available result rows and prepare comparison figures. This section prepares evidence only.


In [ ]:
comparison = create_final_comparison(paths["outputs"], paths["figures"], wandb_config=CONFIG.wandb)
display(comparison.sort_values("macro_f1", ascending=False))


## 17. Error Analysis Evidence

For the best available model, save confusion-matrix evidence and example predictions. Do not interpret the results here.

TODO: Inspect the saved JSON and figures before writing your own analysis separately.


In [ ]:
error_analysis = create_error_analysis(paths["outputs"], paths["figures"], wandb_config=CONFIG.wandb)
print(f"Best model: {error_analysis['best_model']['model_name']}")
print(f"Saved confusion matrix: {error_analysis['confusion_matrix_path']}")

display(pd.DataFrame(error_analysis["top_confused_label_pairs"]).head(10))
display(pd.DataFrame(error_analysis["correct_prediction_examples"]).head(5))
display(pd.DataFrame(error_analysis["incorrect_prediction_examples"]).head(5))
